# Milestone 2 — Skill Replay + Manual Command

**Project**: PBT Meta-Controller (Hierarchical PBT, Phase 1)
**Goal**: Run the M1 walking skill at 50 Hz controlled by interactive sliders, with the SharedCommandBuffer interface in place for M3.

---

## What this notebook does

1. Load the M1 trained checkpoint (`m1_v2_run1/model_249.pt`)
2. Build a fresh Genesis env with **n_envs=1** (single robot, real-time semantics)
3. Set up `SharedCommandBuffer` + ipywidgets sliders that write to it
4. Run `FastLoop` for 5 minutes (15,000 steps @ 50 Hz)
5. Display live visualization (frame every 25 steps ≈ 2 Hz visual update)
6. Save:
   - `logs/m2_run_<timestamp>.jsonl` — every step
   - `logs/m2_run_<timestamp>.mp4` — video of all rendered frames

---

## Architecture for M2

```
                  main thread
                       │
       ┌───────────────┼───────────────┐
       │               │               │
       ▼               ▼               ▼
   sliders     SharedCommandBuffer    FastLoop
   (ipywidgets)    (lock + dict)      (50 Hz)
       │               ▲               │
       └───── set ─────┘               │
                       │               │
                       └─── get ───────┘
```

The FastLoop and the slider callbacks run in the same Python process, but the
buffer's lock is real — ipywidgets callbacks may execute on the kernel's event
loop thread. The lock means M3 can swap "slider writes buffer" for "VLM writes
buffer" with no FastLoop changes.

---

## Prereq

- M0 PASS (Genesis + Ollama installed)
- M1 PASS (checkpoint at `pbt_meta/skill/checkpoints/m1_v2_run1/model_249.pt`)
- Same Colab session as M0/M1 OR fresh session (notebook re-installs deps)

**Time budget**: ~10 min (build env 1 min + run 5 min + post-process 1 min)

## Step 1 — Install deps + verify modules

In [ ]:
# Step 1: Install deps + import pbt_meta modules
import subprocess
import sys

print("=" * 60)
print("STEP 1: Install + verify")
print("=" * 60)

# These should already be installed from M0/M1, but be defensive on fresh sessions
for pkg in ["genesis-world==0.4.3", "rsl-rl-lib==5.0.0", "ipywidgets", "imageio[ffmpeg]"]:
    result = subprocess.run(
        ["pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"[FAIL] {pkg}")
        print(result.stderr[-300:])
    else:
        print(f"[OK] {pkg}")

# Add repo root to path so we can import pbt_meta
import os
REPO_ROOT = "/content/hierarchical-pbt"
if not os.path.exists(REPO_ROOT):
    print()
    print(f"[FAIL] Repo not found at {REPO_ROOT}")
    print("       Clone with: !git clone https://github.com/<USER>/hierarchical-pbt.git /content/hierarchical-pbt")
    raise SystemExit(1)

sys.path.insert(0, REPO_ROOT)

# Verify imports
from pbt_meta.config import (
    OBS_DIM, ACTION_DIM, CONTROL_DT, CONTROL_HZ,
    COMMAND_RANGES, DEFAULT_SKILL_CHECKPOINT,
)
from pbt_meta.loop.shared_state import SharedCommandBuffer
from pbt_meta.loop.fast import FastLoop, FastLoopStats

print()
print(f"[OK] pbt_meta imported")
print(f"  OBS_DIM = {OBS_DIM}")
print(f"  ACTION_DIM = {ACTION_DIM}")
print(f"  CONTROL_HZ = {CONTROL_HZ}")
print(f"  COMMAND_RANGES = {COMMAND_RANGES}")

## Step 2 — Init Genesis (idempotent)

In [ ]:
# Step 2: Initialize Genesis
import genesis as gs

try:
    gs.init(backend=gs.gpu, precision="32", logging_level="warning")
    print("[OK] Genesis initialized fresh")
except Exception as e:
    if "already" in str(e).lower() or "initialized" in str(e).lower():
        print("[OK] Genesis already initialized (from prior notebook in this session)")
    else:
        raise

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

free, total = torch.cuda.mem_get_info()
print(f"GPU: {(total-free)/1e9:.1f} / {total/1e9:.1f} GB used ({free/1e9:.1f} GB free)")

## Step 3 — Build env with n_envs=1

We use the same `go2_env.py` and patched config as M1, but build with `num_envs=1` for real-time single-robot semantics. This rebuilds the env (~10 sec).

In [ ]:
# Step 3: Build env (n_envs=1)
import os
import sys
import importlib
import gc
import time
import torch

# Genesis Go2 example files (downloaded in M1, should still be at /content/go2_train)
GO2_DIR = "/content/go2_train"
if not os.path.exists(os.path.join(GO2_DIR, "go2_env.py")):
    print(f"[INFO] Re-downloading go2 files to {GO2_DIR}")
    os.makedirs(GO2_DIR, exist_ok=True)
    os.chdir(GO2_DIR)
    import urllib.request
    GENESIS_BASE = "https://raw.githubusercontent.com/Genesis-Embodied-AI/Genesis/main/examples/locomotion"
    for fname in ["go2_env.py", "go2_train.py", "go2_eval.py"]:
        urllib.request.urlretrieve(f"{GENESIS_BASE}/{fname}", fname)
        print(f"  [OK] {fname}")

sys.path.insert(0, GO2_DIR)

# Clear any existing env from prior cell runs
try:
    del env
except NameError:
    pass
try:
    del runner
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# Apply same patches as M1 v2
import go2_train
importlib.reload(go2_train)

original_get_cfgs = go2_train.get_cfgs

def patched_get_cfgs():
    env_cfg, obs_cfg, reward_cfg, command_cfg = original_get_cfgs()
    command_cfg["lin_vel_x_range"] = [-1.0, 1.0]
    command_cfg["lin_vel_y_range"] = [-0.5, 0.5]
    command_cfg["ang_vel_range"]   = [-1.0, 1.0]
    reward_cfg["reward_scales"]["tracking_ang_vel"] = 1.0
    return env_cfg, obs_cfg, reward_cfg, command_cfg

go2_train.get_cfgs = patched_get_cfgs

env_cfg, obs_cfg, reward_cfg, command_cfg = go2_train.get_cfgs()

# Build env with N=1
from go2_env import Go2Env

NUM_ENVS = 1
print(f"Building env with num_envs={NUM_ENVS}...")
t0 = time.time()
env = Go2Env(
    num_envs=NUM_ENVS,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    show_viewer=False,
)
print(f"[OK] Built in {time.time()-t0:.1f}s")

free, total = torch.cuda.mem_get_info()
print(f"GPU after build: {free/1e9:.1f} GB free")

## Step 4 — Add a camera + load checkpoint

The camera is added separately from the env build. The Genesis camera renders
the same scene as the env, so we get visualizations of robot 0.

In [ ]:
# Step 4: Add camera + load checkpoint
import os
import torch
from rsl_rl.runners import OnPolicyRunner

print("=" * 60)
print("STEP 4: Camera + checkpoint")
print("=" * 60)

# --- Camera ---
# IMPORTANT: in Genesis, cameras are added to scene BEFORE build. But our env
# already built. We work around this by accessing env.scene if exposed.
camera = None
if hasattr(env, "scene"):
    try:
        camera = env.scene.add_camera(
            res=(640, 480),
            pos=(2.5, 0.0, 1.5),
            lookat=(0.0, 0.0, 0.5),
            fov=40,
            GUI=False,
        )
        print("[OK] Camera added to existing scene")
        # Some Genesis versions need a build call after add_camera
        # Try it; if it fails, the camera may still work
        try:
            env.scene.build_camera(camera)
            print("[OK] Camera built")
        except (AttributeError, Exception):
            pass
    except Exception as e:
        print(f"[WARN] Could not add camera: {e}")
        print("       FastLoop will run without rendering")
        camera = None
else:
    print("[WARN] env has no .scene attribute — no rendering")

# --- Checkpoint ---
CKPT = "/content/hierarchical-pbt/pbt_meta/skill/checkpoints/m1_v2_run1/model_249.pt"
assert os.path.exists(CKPT), f"Checkpoint not found at {CKPT}"
print(f"\nLoading checkpoint: {CKPT}")

train_cfg = go2_train.get_train_cfg("m2_skill_replay")  # log dir name
log_dir = "/content/m2_runner_log"
os.makedirs(log_dir, exist_ok=True)

runner = OnPolicyRunner(env, train_cfg, log_dir, device="cuda:0")
runner.load(CKPT)
print(f"[OK] Loaded weights")

policy = runner.get_inference_policy(device="cuda:0")
print(f"[OK] Got inference policy")

# Quick sanity: 10 zero-command steps to confirm robot is stable
print("\nSanity: 10 stop-command steps...")
with torch.inference_mode():
    obs_td = env.reset()
    for _ in range(10):
        env.commands[:, 0] = 0.0
        env.commands[:, 1] = 0.0
        env.commands[:, 2] = 0.0
        actions = policy(obs_td)
        step_res = env.step(actions)
        obs_td = step_res[0] if isinstance(step_res, tuple) else step_res
    h = env.base_pos[0, 2].item()
    print(f"  height after 10 steps: {h:.3f}m (should be ~0.32)")

## Step 5 — Set up SharedCommandBuffer + sliders

In [ ]:
# Step 5: Buffer + sliders + control panel
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create the buffer
buffer = SharedCommandBuffer()

# Sliders
slider_vx = widgets.FloatSlider(
    value=0.0, min=-1.0, max=1.0, step=0.1,
    description="vx (m/s):", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)
slider_vy = widgets.FloatSlider(
    value=0.0, min=-0.5, max=0.5, step=0.05,
    description="vy (m/s):", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)
slider_wz = widgets.FloatSlider(
    value=0.0, min=-1.0, max=1.0, step=0.1,
    description="wz (rad/s):", continuous_update=True,
    layout=widgets.Layout(width="500px"),
)

# Buttons
btn_stop_now = widgets.Button(description="STOP", button_style="warning")
btn_zero_sliders = widgets.Button(description="Zero all", button_style="info")

def on_slider_change(change):
    buffer.set(slider_vx.value, slider_vy.value, slider_wz.value)

slider_vx.observe(on_slider_change, names="value")
slider_vy.observe(on_slider_change, names="value")
slider_wz.observe(on_slider_change, names="value")

def zero_all(_):
    slider_vx.value = 0.0
    slider_vy.value = 0.0
    slider_wz.value = 0.0
    # Slider observe will fire and call buffer.set

def stop_now(_):
    buffer.stop()  # immediate, doesn't wait for slider observe
    slider_vx.value = 0.0
    slider_vy.value = 0.0
    slider_wz.value = 0.0

btn_zero_sliders.on_click(zero_all)
btn_stop_now.on_click(stop_now)

print("Sliders ready. Try moving them — buffer should update.")
print(f"Initial buffer state: {buffer.get()}")
display(widgets.VBox([
    slider_vx, slider_vy, slider_wz,
    widgets.HBox([btn_stop_now, btn_zero_sliders]),
]))

# Test: write something via slider, verify buffer
slider_vx.value = 0.5
print(f"After setting vx=0.5: buffer = {buffer.get()}")
slider_vx.value = 0.0
print(f"After resetting: buffer = {buffer.get()}")
print(f"Buffer stats: {buffer.stats()}")

## Step 6 — Live visualization setup

We display the most recent rendered frame in an output widget. The FastLoop
calls `on_step` after each step; if a new frame was rendered, we update the
display.

In [ ]:
# Step 6: Live visualization
import io
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets

# Output widget for live image + stats
out_image = widgets.Output()
out_stats = widgets.Output()

# Initial empty display
with out_image:
    print("(no frame yet — start the loop)")
with out_stats:
    print("(no stats yet)")

display(widgets.HBox([out_image, out_stats]))

def make_on_step_callback(loop_ref, render_every=25):
    """Returns a callback closure for FastLoop.run(on_step=...).
    
    Updates the display every `render_every` steps. Cheap when not rendering.
    """
    last_displayed_render = [-1]  # mutable closure var
    
    def callback(step, record):
        # Only update display when there's a new rendered frame
        n_renders = len(loop_ref.rendered_frames)
        if n_renders > last_displayed_render[0]:
            last_displayed_render[0] = n_renders
            
            # Show latest frame
            with out_image:
                clear_output(wait=True)
                if loop_ref.rendered_frames:
                    rgb = loop_ref.rendered_frames[-1]
                    fig, ax = plt.subplots(figsize=(6, 4.5))
                    ax.imshow(rgb)
                    ax.axis("off")
                    ax.set_title(f"step {step}")
                    plt.tight_layout()
                    plt.show()
            
            # Show stats
            with out_stats:
                clear_output(wait=True)
                print(f"step:        {step}")
                print(f"cmd:         vx={record['cmd'][0]:+.2f}  vy={record['cmd'][1]:+.2f}  wz={record['cmd'][2]:+.2f}")
                print(f"actual:      vx={record['actual_vx']:+.2f}  vy={record['actual_vy']:+.2f}  wz={record['actual_wz']:+.2f}" if record['actual_vx'] is not None else f"actual_wz: {record['actual_wz']:+.2f}")
                print(f"height:      {record['height']:.3f} m")
                print(f"step_dt:     {record['step_dt_ms']:.1f} ms")
                print(f"frames:      {n_renders}")
                buf_stats = buffer.stats()
                print(f"buffer:      writes={buf_stats['n_writes']}  reads={buf_stats['n_reads']}  clamps={buf_stats['n_clamps']}")
    
    return callback

print("Visualization widgets ready.")

## Step 7 — Run the FastLoop

The loop runs for `MAX_STEPS` (= 15000 = 5 min @ 50 Hz). While it runs:
- Move the sliders to drive the robot
- Click STOP to halt early
- Watch the live frame + stats panel

After it stops, we save logs + video.

In [ ]:
# Step 7: Run the FastLoop
import os
import time
from pathlib import Path

print("=" * 60)
print("STEP 7: FastLoop run")
print("=" * 60)

# Where to save logs (relative to repo so we can commit summaries later)
LOG_DIR = Path("/content/m2_logs")
LOG_DIR.mkdir(exist_ok=True)

timestamp = time.strftime("%Y%m%d_%H%M%S")
log_path = LOG_DIR / f"m2_run_{timestamp}.jsonl"

# Loop config
MAX_STEPS = 15000  # 5 min @ 50 Hz
RENDER_EVERY = 25  # 0.5 sec real-time

print(f"Log file:    {log_path}")
print(f"Max steps:   {MAX_STEPS}  ({MAX_STEPS / 50 / 60:.1f} min @ 50 Hz)")
print(f"Render every: {RENDER_EVERY} steps  ({RENDER_EVERY / 50:.2f} sec)")
print()
print("→ Move the sliders ABOVE to drive the robot")
print("→ Click STOP to halt early")
print()

# Create the FastLoop
loop = FastLoop(
    env=env,
    policy=policy,
    buffer=buffer,
    log_path=log_path,
    camera=camera,
    render_every_n_steps=RENDER_EVERY,
    device="cuda:0",
)

# Wire stop button to loop
def request_stop(_):
    loop.stop()
    print("[STOP requested — loop will halt at next iteration]")

btn_stop_now.on_click(request_stop)

# Build callback (depends on `loop` reference)
on_step = make_on_step_callback(loop, render_every=RENDER_EVERY)

# RUN
print("Starting loop... (this is blocking, but UI events still fire)")
print()
t_start = time.time()
stats = loop.run(max_steps=MAX_STEPS, on_step=on_step)
elapsed = time.time() - t_start

print()
print("=" * 60)
print("LOOP DONE")
print("=" * 60)
print(f"Steps:         {stats.n_steps}")
print(f"Wall clock:    {stats.wall_clock_sec:.1f} sec")
print(f"Actual FPS:    {stats.actual_fps:.1f}")
print(f"Target FPS:    {stats.target_fps:.1f}")
print(f"FPS ratio:     {stats.fps_ratio():.2%}  (1.0 = perfect)")
print(f"Renders:       {stats.n_renders}")
print(f"Log writes:    {stats.n_log_writes}")
print(f"Buffer stats:  {buffer.stats()}")
print()
print(f"Log file size: {log_path.stat().st_size / 1e6:.2f} MB")

## Step 8 — Save video

In [ ]:
# Step 8: Encode rendered frames to mp4
import imageio
from pathlib import Path

if not loop.rendered_frames:
    print("[SKIP] No frames rendered")
else:
    timestamp_str = log_path.stem.replace("m2_run_", "")
    video_path = LOG_DIR / f"m2_run_{timestamp_str}.mp4"
    
    # Frames were rendered every N steps, so playback at native rate is 50/N fps
    # = 2 fps for N=25. That's slow. Speed up to 10 fps for watchability.
    fps_playback = 10
    
    print(f"Encoding {len(loop.rendered_frames)} frames to {video_path}...")
    print(f"  Playback fps: {fps_playback}")
    print(f"  Wallclock duration: {len(loop.rendered_frames) / fps_playback:.1f} sec")
    print(f"  Real time covered: {len(loop.rendered_frames) * RENDER_EVERY / 50:.1f} sec")
    
    with imageio.get_writer(str(video_path), fps=fps_playback) as writer:
        for frame in loop.rendered_frames:
            writer.append_data(frame)
    
    size_mb = video_path.stat().st_size / 1e6
    print(f"[OK] Saved {video_path}  ({size_mb:.1f} MB)")

## Step 9 — Quick log analysis

Read back the JSONL log and check basic statistics. We don't analyze deeply
here — that's for a separate analysis notebook.

In [ ]:
# Step 9: Analyze log
import json
import numpy as np
from pathlib import Path

print("=" * 60)
print("STEP 9: Log analysis")
print("=" * 60)

records = []
with open(log_path) as f:
    for line in f:
        records.append(json.loads(line))

print(f"Loaded {len(records)} records from {log_path.name}")

if not records:
    print("[WARN] Empty log")
else:
    # Extract step dt distribution
    step_dts = np.array([r["step_dt_ms"] for r in records])
    print()
    print(f"Step dt (ms):")
    print(f"  mean:   {step_dts.mean():.2f}")
    print(f"  median: {np.median(step_dts):.2f}")
    print(f"  std:    {step_dts.std():.2f}")
    print(f"  p99:    {np.percentile(step_dts, 99):.2f}")
    print(f"  max:    {step_dts.max():.2f}")
    print()
    print(f"Target dt: {1000/50:.1f} ms (50 Hz)")
    
    n_over_target = (step_dts > 1000/50).sum()
    print(f"Steps over target: {n_over_target} / {len(records)} ({n_over_target/len(records):.1%})")
    
    # Command stats
    cmds = np.array([r["cmd"] for r in records])
    print()
    print(f"Command ranges in run:")
    print(f"  vx: [{cmds[:, 0].min():+.2f}, {cmds[:, 0].max():+.2f}]")
    print(f"  vy: [{cmds[:, 1].min():+.2f}, {cmds[:, 1].max():+.2f}]")
    print(f"  wz: [{cmds[:, 2].min():+.2f}, {cmds[:, 2].max():+.2f}]")
    
    # Height stats (fall detection)
    heights = np.array([r["height"] for r in records])
    n_fall = (heights < 0.2).sum()
    print()
    print(f"Height stats:")
    print(f"  mean: {heights.mean():.3f} m")
    print(f"  min:  {heights.min():.3f} m")
    print(f"  max:  {heights.max():.3f} m")
    print(f"  Steps with h<0.2 (fall): {n_fall} ({n_fall/len(records):.1%})")

## Step 10 — Summary

In [ ]:
# Step 10: M2 PASS check
import torch

print("=" * 60)
print("M2 SUMMARY")
print("=" * 60)

# Pass criteria from milestones.md
print("PASS criteria:")
print(f"  [{'OK' if stats.actual_fps >= 45 else 'FAIL'}] Fast loop @ 50 Hz stable (actual {stats.actual_fps:.1f} Hz, threshold 45)")
print(f"  [{'OK' if stats.n_steps >= 5000 else 'FAIL'}] No crash after 5 min (got {stats.n_steps} steps, threshold 5000 = ~1.7 min minimum)")
print(f"  [{'OK' if log_path.stat().st_size > 0 else 'FAIL'}] Log file usable ({log_path.stat().st_size / 1e6:.2f} MB)")
print(f"  [N/A]  Slider response within 1-2 steps (visual check during run)")
print()

free, total = torch.cuda.mem_get_info()
print(f"GPU: {(total-free)/1e9:.1f} / {total/1e9:.1f} GB used")
print()
print("Files:")
print(f"  Log:   {log_path}")
if 'video_path' in dir() and video_path.exists():
    print(f"  Video: {video_path}")
print()
print("Next: M3 — VLM dispatch (replace slider with Qwen2.5-VL-3B)")

## What to report back

1. **Step 3** — env build time + GPU memory after build (n_envs=1 should be tiny, ~50 MB)
2. **Step 4** — did `add_camera` work after env build? If not, video will be empty
3. **Step 7** — `actual_fps` from stats output. **This is the critical M2 metric.**
4. **Step 9** — step_dt p99 + n_over_target percentage
5. **Subjective** — did the sliders feel responsive? (1-2 step delay = good, 5+ step delay = problem)

If `actual_fps` < 45 → debug per-step overhead. Most likely culprits:
- Camera render in main loop blocking the next step
- Tensor → numpy → list conversions for log serialization
- ipywidgets display update inside callback

If sliders feel laggy but FPS is good → the issue is in the display refresh, not the loop itself.